In [1]:
import pandas as pd

In [10]:
df = pd.read_excel('sheets/attendance_202601.xlsx')
df = df[df['OK'] == 1]
df["Date"] = [str(item)[:10] for item in df["Kezdés ideje"].values]
df["Neptun code"] = df["Neptun code"].apply(lambda x: x.upper())
# df = df.groupby(['Date', 'Class']).size().reset_index(name='Count')
# df = df.sort_values(by='Neptun code', ascending=True)
# df.rename(columns={"Count":"Count of participations"}, inplace=True)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 431 entries, 1 to 442
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   430 non-null    float64       
 1   Kezdés ideje         430 non-null    datetime64[ns]
 2   Befejezés időpontja  431 non-null    datetime64[ns]
 3   E-mail-cím           431 non-null    object        
 4   Név                  431 non-null    object        
 5   Name                 431 non-null    object        
 6   Neptun code          431 non-null    object        
 7   Class                431 non-null    object        
 8   OK                   431 non-null    int64         
 9   Date                 431 non-null    object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(6)
memory usage: 37.0+ KB


,Id,Kezdés ideje,Befejezés időpontja,E-mail-cím,Név,Name,Neptun code,Class,OK,Date
1,1069.0,2026-02-09 16:10:00,2026-02-09 16:10:30,lalahh@mailbox.unideb.hu,Szabó László Márk,Szabó László Márk,IVUC6J,AI - Monday - 16:00-18:00 (IK-204),1,2026-02-09
2,1070.0,2026-02-09 16:10:09,2026-02-09 16:10:32,szabolilihanna@mailbox.unideb.hu,Szabó Lili Hanna,Szabó Lili Hanna,RP88LS,AI - Monday - 16:00-18:00 (IK-204),1,2026-02-09
3,1071.0,2026-02-09 16:10:06,2026-02-09 16:10:33,d.rekaaaa@mailbox.unideb.hu,Dagonya Réka,Dagonya Réka,B906KU,AI - Monday - 16:00-18:00 (IK-204),1,2026-02-09
4,1072.0,2026-02-09 16:10:14,2026-02-09 16:10:34,gerelly@mailbox.unideb.hu,Popovics Gergő,Popovics Gergő,AABZM4,AI - Monday - 16:00-18:00 (IK-204),1,2026-02-09
5,1073.0,2026-02-09 16:10:26,2026-02-09 16:10:50,gdominik2004@mailbox.unideb.hu,Goda Dominik,Goda Dominik,X2LSLL,AI - Monday - 16:00-18:00 (IK-204),1,2026-02-09


In [11]:
neptun_codes_by_class = df.groupby('Class')['Neptun code'].apply(list).to_dict()
print(neptun_codes_by_class)

{'AI - Monday - 16:00-18:00 (IK-204)': ['IVUC6J', 'RP88LS', 'B906KU', 'AABZM4', 'X2LSLL', 'GWZENN', 'S4BMSQ', 'F7DBHE', 'MHH6B8', 'YOYC0C', 'S41OQA', 'CXNOLK', 'AV2KLX', 'IR9X23', 'DMFZNH', 'E8PIZS', 'S4BMSQ', 'B906KU', 'KCRMSH', 'RP88LS', 'DMFZNH', 'S41OQA', 'AABZM4', 'GWZENN', 'YOYC0C', 'CXNOLK', 'CTDXKK', 'X2LSLL', 'E8PIZS', 'F7DBHE', 'IVUC6J', 'AV2KLX', 'IR9X23', 'WUEXFL', 'S4BMSQ', 'X2LSLL', 'CTDXKK', 'CXNOLK', 'GWZENN', 'KCRMSH', 'S41OQA', 'E8PIZS', 'B906KU', 'RP88LS', 'DMFZNH', 'IR9X23', 'F7DBHE', 'AV2KLX', 'S4BMSQ', 'KCRMSH', 'B906KU', 'CXNOLK', 'CTDXKK', 'S41OQA', 'IVUC6J', 'AABZM4', 'RP88LS', 'GWZENN', 'E8PIZS', 'X2LSLL', 'MHH6B8', 'AV2KLX', 'YOYC0C', 'S41OQA', 'F7DBHE', 'CXNOLK', 'S4BMSQ', 'X2LSLL', 'AV2KLX', 'B906KU', 'DMFZNH', 'MHH6B8', 'KCRMSH', 'YOYC0C', 'RP88LS', 'E8PIZS', 'AABZM4', 'IR9X23', 'DMFZNH', 'YOYC0C', 'S41OQA', 'S4BMSQ', 'KCRMSH', 'IR9X23', 'IVUC6J', 'E8PIZS', 'X2LSLL', 'AABZM4', 'AV2KLX', 'RP88LS', 'CTDXKK', 'X2LSLL', 'CXNOLK', 'KCRMSH', 'AABZM4', 'CTDXKK', 

In [29]:
from collections import defaultdict
from openpyxl import load_workbook

absent_by_class_and_date = defaultdict(dict)

for class_name, all_neptun_codes in neptun_codes_by_class.items():
    class_df = df[df['Class'] == class_name]
    for date in class_df['Date'].unique():
        present_codes = set(class_df[class_df['Date'] == date]['Neptun code'])
        absent_codes = set(all_neptun_codes) - present_codes
        absent_by_class_and_date[class_name][date] = {
            "missed": list(absent_codes),
            "present" : list(present_codes)}

# Példa kiírás
def add_student_info(neptun_code, status, dict):
    dict["Hallgató neve"].append("")
    dict["Neptunkód"].append(neptun_code)
    dict["Képzés"].append("")
    dict["Jelenlét"].append(status)

# 202601/{class_name}_{date}.txt
for class_name, dates in absent_by_class_and_date.items():    
    for date, absent_codes in dates.items():
        # dict_status = { "Hallgató neve" : [], "Neptunkód" : [], "Képzés" : [], "Jelenlét" : [] }
        wb = load_workbook('sheets/tmp.xlsx')
        ws = wb.get_sheet_by_name("Neptun export")

        print(f"{class_name} | {date}")
        missed = absent_codes["missed"]
        
        idx = 1
        for m in missed: 
            # add_student_info(m, "Nem jelent meg", dict_status)
            ws.cell(row=idx, column=2).value = m
            ws.cell(row=idx, column=4).value = "Nem jelent meg"
            idx += 1
            
        print(f"Hiányzók: {missed}")
        present = absent_codes["present"]
        print(f"Jelenlévő: {present}\n")

        for p in present: 
            # add_student_info(p, "Megjelent", dict_status)
            ws.cell(row=idx, column=2).value = p
            ws.cell(row=idx, column=4).value = "Megjelent"
            idx += 1
        
        dict_status = pd.DataFrame(dict_status)
        fn = f"sheets/202601/{class_name}_{date}.xlsx"
        fn = fn.replace(" ", "-").replace("(", "").replace(")", "").replace(":", "")
        print(dict_status)
        # dict_status.to_excel(fn, index=False,sheet_name=f"Neptun export")
        wb.save(fn)
        print("-----\n")      

C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")


AI - Monday - 16:00-18:00 (IK-204) | 2026-02-09
Hiányzók: ['CTDXKK', 'KCRMSH']
Jelenlévő: ['CXNOLK', 'E8PIZS', 'YOYC0C', 'IVUC6J', 'IR9X23', 'S41OQA', 'AV2KLX', 'DMFZNH', 'F7DBHE', 'X2LSLL', 'MHH6B8', 'B906KU', 'S4BMSQ', 'AABZM4', 'WUEXFL', 'RP88LS', 'GWZENN']

   Hallgató neve Neptunkód Képzés        Jelenlét
0                   WVNL1I         Nem jelent meg
1                   JMQNBG         Nem jelent meg
2                   BF2563         Nem jelent meg
3                   PLXFKZ              Megjelent
4                   GHXSXP              Megjelent
5                   F91DW3              Megjelent
6                   IQ3DUH              Megjelent
7                   JUE213              Megjelent
8                   KF82GR              Megjelent
9                  WVNL1I               Megjelent
10                  QKWED0              Megjelent
11                  HLIVC2              Megjelent
12                  F16RVU              Megjelent
13                  V7KZIX            

C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")


-----

AI - Monday - 18:00-20:00 (IK-204) | 2026-04-13
Hiányzók: ['08KJR7', 'F9Z2WO', 'CD56LX', 'ALD4X0', 'FIPUIJ', 'CYF65U', 'JP15KJ']
Jelenlévő: ['BMLPAG', 'C9YUAV', 'TUSGQV', 'O8KJR7', 'MIL8I8', 'SQAEL4', 'WUEXFL', 'RG1BUN', 'RVPNH2']

   Hallgató neve Neptunkód Képzés        Jelenlét
0                   WVNL1I         Nem jelent meg
1                   JMQNBG         Nem jelent meg
2                   BF2563         Nem jelent meg
3                   PLXFKZ              Megjelent
4                   GHXSXP              Megjelent
5                   F91DW3              Megjelent
6                   IQ3DUH              Megjelent
7                   JUE213              Megjelent
8                   KF82GR              Megjelent
9                  WVNL1I               Megjelent
10                  QKWED0              Megjelent
11                  HLIVC2              Megjelent
12                  F16RVU              Megjelent
13                  V7KZIX              Megjelent
-----

AI -

C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")


-----

ANLP - Tuesday - 18:00-20:00 (IK-TEOKJ II.em 106) | 2026-03-17
Hiányzók: ['M1VRXK', 'HJTTN3', 'EJSJ2L']
Jelenlévő: ['POOHS5', 'B167PH', 'K0B7PM', 'JHB2VY', 'YUTDI1', 'SPJWSY', 'Z6OAK3', 'B8HLCC', 'QCBUQN', 'O3NNV6', 'B51UHJ', 'TEWLCO']

   Hallgató neve Neptunkód Képzés        Jelenlét
0                   WVNL1I         Nem jelent meg
1                   JMQNBG         Nem jelent meg
2                   BF2563         Nem jelent meg
3                   PLXFKZ              Megjelent
4                   GHXSXP              Megjelent
5                   F91DW3              Megjelent
6                   IQ3DUH              Megjelent
7                   JUE213              Megjelent
8                   KF82GR              Megjelent
9                  WVNL1I               Megjelent
10                  QKWED0              Megjelent
11                  HLIVC2              Megjelent
12                  F16RVU              Megjelent
13                  V7KZIX              Megjelent
-----


C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")


-----

ANLP - Tuesday - 14:00-16:00 (IK-132) | 2026-02-24
Hiányzók: ['BF2563', 'WVNL1I ']
Jelenlévő: ['PLXFKZ', 'GHXSXP', 'F91DW3', 'JMQNBG', 'IQ3DUH', 'JUE213', 'KF82GR', 'HLIVC2', 'F16RVU', 'WVNL1I', 'QKWED0', 'V7KZIX']

   Hallgató neve Neptunkód Képzés        Jelenlét
0                   WVNL1I         Nem jelent meg
1                   JMQNBG         Nem jelent meg
2                   BF2563         Nem jelent meg
3                   PLXFKZ              Megjelent
4                   GHXSXP              Megjelent
5                   F91DW3              Megjelent
6                   IQ3DUH              Megjelent
7                   JUE213              Megjelent
8                   KF82GR              Megjelent
9                  WVNL1I               Megjelent
10                  QKWED0              Megjelent
11                  HLIVC2              Megjelent
12                  F16RVU              Megjelent
13                  V7KZIX              Megjelent
-----

ANLP - Tuesday - 14:

C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
C:\Users\lrobe\AppData\Local\Temp\ipykernel_16036\1313520907.py:27: DeprecationWarning: Call to deprecated function get_sheet_by_name (Use wb[sheetname]).
  ws = wb.get_sheet_by_name("Neptun export")
